# 12. Bias-Variance Tradeoff — Choosing the System's Final Complexity

**Building a Heart Disease Risk-Screening System — Notebook 12 of 12, Stage 5: Building and Validating the Predictive Core**

Notebook 11 showed *that* an overly flexible tree overfits this registry. This
closing notebook answers *why*, in a way that generalizes to any model choice —
not just tree depth, but the choice between a simple model and a complex one for
any system, anywhere. That answer is what finally lets us pick, with justification
rather than guesswork, how complex the deployed risk-screening model should be.

## The topic

$$\text{Expected error} = \underbrace{\text{Bias}^2}_{\text{too simple}} + \underbrace{\text{Variance}}_{\text{too sensitive to the training sample}} + \underbrace{\text{Irreducible error}}_{\text{noise no model can remove}}$$

**Bias** is error from a model too rigid to capture the real
age/cholesterol/chest-pain-to-disease relationship — it under-predicts and
over-predicts in the same systematic places no matter how much data it sees.
**Variance** is error from a model so flexible it fits noise specific to whichever
350 patients happened to be in the training split — retrain on a different split
and you'd get a meaningfully different model.

## Why it matters for this system

This is the final decision this module builds toward: not "is more complexity
better" (Notebook 11 already showed it isn't, past a point) but *how much*
complexity this specific registry, at this specific size, can support — a question
with a precise, checkable answer instead of a rule of thumb.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
inputs = ["age", "sex", "cp", "trestbps", "chol", "thalach", "exang"]
X, y = df[inputs].to_numpy(), df["target"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## The toolkit

| Tool | Reveals |
|---|---|
| **Bootstrap resampling** | How much predictions for the SAME patient vary across different training samples — variance, made measurable |
| **Averaging bootstrap predictions** | Cancels out variance, exposing bias directly |
| **Complexity sweep + decomposition** | The full bias/variance/total-error picture across a range of settings |
| **Ensembling (e.g. Random Forest)** | A direct variance-reduction technique, once you know variance is the dominant problem |

## How to choose

Run the bootstrap-based decomposition specifically when you need to know *which*
lever to pull — more regularization (if variance dominates) or a fundamentally
different, more flexible model family (if bias dominates), rather than guessing.
Reach for an ensemble method once the decomposition confirms variance is the
bigger problem for your current model family — that's precisely the situation
ensembling is built to address, by averaging away the sample-to-sample sensitivity
a single model exhibits.

## Applied to the registry

### Measuring variance directly: same model family, different training samples

In [ ]:
rng = np.random.default_rng(0)
n_bootstrap = 30

def bootstrap_predictions(max_depth):
    all_predictions = []
    for i in range(n_bootstrap):
        idx = rng.integers(0, len(X_train), len(X_train))
        tree = DecisionTreeClassifier(max_depth=max_depth, random_state=0)
        tree.fit(X_train[idx], y_train[idx])
        all_predictions.append(tree.predict_proba(X_test)[:, 1])
    return np.array(all_predictions)  # shape: (n_bootstrap, n_test_patients)

shallow_predictions = bootstrap_predictions(max_depth=2)
deep_predictions = bootstrap_predictions(max_depth=None)
print("Shape (n_bootstrap_models, n_test_patients):", shallow_predictions.shape)

### Variance: how much do predictions for the SAME patient disagree?

In [ ]:
patient_idx = 0
print(f"Predicted disease probability for test patient #{patient_idx} (actual outcome: {y_test[patient_idx]}):")
print(f"  Shallow tree (depth=2): mean={shallow_predictions[:,patient_idx].mean():.3f}, "
      f"std={shallow_predictions[:,patient_idx].std():.3f}")
print(f"  Deep tree (unlimited):  mean={deep_predictions[:,patient_idx].mean():.3f}, "
      f"std={deep_predictions[:,patient_idx].std():.3f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(shallow_predictions[:,patient_idx], bins=15, alpha=0.6, label="shallow (depth=2)", color="steelblue")
ax.hist(deep_predictions[:,patient_idx], bins=15, alpha=0.6, label="deep (unlimited)", color="darkorange")
ax.axvline(y_test[patient_idx], color="black", linestyle="--", label="actual outcome")
ax.set_xlabel("predicted probability of disease"); ax.legend()
ax.set_title(f"Spread of predictions for one patient across {n_bootstrap} bootstrap samples")
plt.show()

### Bias: how far off is the AVERAGE prediction?

Averaging cancels out variance, leaving mostly bias — a systematic gap here means
the model type itself is too rigid for this relationship, independent of which
training sample it saw.

In [ ]:
shallow_avg = shallow_predictions.mean(axis=0)
deep_avg = deep_predictions.mean(axis=0)

shallow_bias = np.mean(np.abs(shallow_avg - y_test))
deep_bias = np.mean(np.abs(deep_avg - y_test))
print(f"Shallow tree -- average |bias|: {shallow_bias:.3f}")
print(f"Deep tree    -- average |bias|: {deep_bias:.3f}")

### The full decomposition, and where it points

In [ ]:
def bias_variance_decomposition(predictions_matrix, y_true):
    avg_prediction = predictions_matrix.mean(axis=0)
    bias_squared = np.mean((avg_prediction - y_true) ** 2)
    variance = np.mean(predictions_matrix.var(axis=0))
    total_error = np.mean((predictions_matrix - y_true) ** 2)
    return bias_squared, variance, total_error

depths = [1, 2, 3, 4, 5, 7, 10, None]
depth_labels = [str(d) if d else "unlimited" for d in depths]
bias_sq_list, var_list, total_list = [], [], []

for depth in depths:
    preds = bootstrap_predictions(depth)
    b, v, t = bias_variance_decomposition(preds, y_test)
    bias_sq_list.append(b); var_list.append(v); total_list.append(t)

fig, ax = plt.subplots(figsize=(9, 5))
x_pos = range(len(depths))
ax.plot(x_pos, bias_sq_list, "o-", label="Bias²", color="steelblue")
ax.plot(x_pos, var_list, "o-", label="Variance", color="darkorange")
ax.plot(x_pos, total_list, "o-", label="Total error", color="black", lw=2)
ax.set_xticks(x_pos); ax.set_xticklabels(depth_labels)
ax.set_xlabel("max_depth"); ax.legend()
ax.set_title("Bias falls, variance rises, as tree depth increases")
plt.show()

best_idx = int(np.argmin(total_list))
print(f"Lowest total error at max_depth = {depth_labels[best_idx]}")

### Ensembling: attacking variance directly, once it's confirmed as the problem

If variance dominates at higher depths (typical for a registry this size), a
Random Forest — many deep trees, each trained on a bootstrap sample, averaged —
is a direct, principled response: it keeps each tree's low bias while averaging
away their individual variance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

single_deep_tree = DecisionTreeClassifier(max_depth=None, random_state=0).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=0).fit(X_train, y_train)

for name, m in [("Single deep tree", single_deep_tree), ("Random Forest (200 trees)", forest)]:
    auc = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
    print(f"{name:28} test AUC = {auc:.3f}")

## Systems view — closing the loop

This is the last checkpoint of the pipeline this module has been building since
Notebook 1: understand the inputs (Stage 1), design trustworthy collection (Stage
2), map relationships (Stage 3), validate claims (Stage 4), and now build and
honestly validate the predictive core (Stage 5). The output this whole system
produces — a probability of disease for a new patient — is exactly the quantity
Notebook 1 opened by teaching you to interpret correctly. Every stage in between
exists to make sure that final probability can actually be trusted.

## Try it yourself

1. Re-run the decomposition using only `age` and `sex` as inputs — does the
   bias/variance balance shift compared to using all 7 validated inputs?
2. Increase `n_bootstrap` from 30 to 100 — does the total-error curve's minimum
   shift, or does the earlier estimate hold up?
3. Compare the Random Forest's own bias/variance decomposition (bootstrap the
   forest the same way trees were bootstrapped above) against the single deep
   tree's — does the forest's variance curve sit lower at every setting, as the
   toolkit section predicts it should?

## Closing the module

Across these twelve notebooks, one thread kept reappearing: a single number — a
probability, a correlation, a p-value, an R², a training accuracy — is never the
full story on its own. Notebook 1 showed a screening test's "90% accurate" hides a
very different real answer depending on prevalence; Notebook 5 showed a pooled
correlation can hide a Simpson's-paradox reversal; this notebook shows a model's
raw performance number hides *why* it performs that way, and whether that
performance will hold up on the next patient. That's the transferable skill this
project was built to teach — not the formulas themselves, but the habit of asking
what a number is quietly assuming before a real clinical decision gets built on
top of it.